# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Juan Sebastian Diaz Campos, 2026

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [1]:
import os
import pandas as pd
import openpyxl

In [2]:
cwd = os.getcwd() # current working directory
raw_dir = cwd + '/../data/raw/'
landing_dir = cwd + '/../data/landing/'

1. Recorriendo los diferentes directorios y sacando los nombres de los archivos:

In [3]:
archivos = []
for ruta in os.listdir(raw_dir):
    if os.path.isdir(raw_dir + ruta):
        for archivo in os.listdir(raw_dir + ruta):
            # validar si es Excel?
            archivos.append(raw_dir + ruta + '/' + archivo)

In [4]:
archivos

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/matrices/Matriz Roles y Perfiles AdminBusiness V8.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/rrhh/1 - HAM_24_Acuerdo_Empleados_Activos.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/rrhh/Colaboradores Migrados a IDM.xlsx']

2. Separando los diferentes archivos para consolidar cada proceso:

In [65]:
archivo_matriz = None
archivo_empleados_idm = None
archivo_empleados_rrhh = None

for archivo in archivos:
    nombre = archivo.lower()

    if "matriz" in nombre:
        archivo_matriz = archivo
    elif "idm" in nombre:
        archivo_empleados_idm = archivo
    elif "ham" in nombre:
        archivo_empleados_rrhh = archivo

3. Inspeccionando los archivos de matrices debido a sus múltiples hojas:

In [32]:
xls = pd.ExcelFile(archivo_matriz)
xls.sheet_names

['Roles VS Cargos', 'Opciones VS Roles']

In [56]:
for hoja in xls.sheet_names:
    df = pd.read_excel(archivo_matriz,sheet_name=hoja,header=None)
    print(hoja)
    display(df.head(15))

Roles VS Cargos


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Matriz de Roles VS Cargos - AdminBusiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NRO. MATRIZ,PERFIL,COD PERFIL,TIPO ENTIDAD,COD CARGO,CARGO,COD DIVISION,DIVISION,COD AREA,AREA,ESTADO,OBSERVACION
4,NaN,V08,ADMINISTRADOR,101,Interno,1000066,INSPECTOR DE CIERRE,1300000,OPERACIONES,1300005,PRODUCCIÓN,Activo,NaN
5,NaN,V08,ADMINISTRADOR,101,Interno,1000061,SUPERVISOR DE PRODUCCIÓN,1300000,OPERACIONES,1300005,PRODUCCIÓN,Activo,NaN
6,NaN,V08,APROBADOR GENERAL,102,Interno,1000089,DIRECTOR GENERAL,1000001,PRESIDENCIA_G1,1000000,DIRECCION GENERAL,Activo,NaN
7,NaN,V08,REVISORIA GLOBAL,103,Interno,1000014,ANALISTA DE COMPRAS,1200000,COMPRAS Y MANTENIMIENTO,1200002,ABASTECIMIENTO,Activo,NaN
8,NaN,V08,REVISORIA GLOBAL,103,Interno,1000085,ANALISTA DE ALMACEN,1300000,OPERACIONES,1300002,LOGÍSTICA Y ALMACENES,Activo,NaN
9,NaN,V08,REVISORIA GLOBAL,103,Interno,1000048,ASISTENTE ADMINISTRATIVO,1100000,SERVICIO AL CLIENTE,1100002,ADMINISTRACIÓN Y SOPORTE,Activo,Incluido reciente por solicitud jefe inmediato


Opciones VS Roles


,0,1,2,3,4,5,6,7,8,9,10
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Matriz de Opciones VS Roles - AdminBusiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COD OPCION,OPCION,DETALLE DE LA OPCION,OPCION SENSIBLE,ADMINISTRADOR,APROBADOR GENERAL,REVISORIA GLOBAL,AUDITORIA,SOPORTE_TI,PREP_FLUJOS_ADM,LANZADOR COMPRAS
4,AB-100,Menú Administración,NaN,NaN,X,X,NaN,NaN,X,NaN,NaN
5,AB-101,Consulta Usuarios,Consultar información de usuarios registrados,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
6,AB-102,Modificar Usuarios,Actualizar usuarios existentes,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
7,AB-103,Asignar Licencia,Asignar licencias de uso a usuarios,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
8,AB-104,Cambio Clave,Restablecer o cambiar contraseña,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
9,AB-105,Crear Festivos,Administrar calendario de días festivos,X,NaN,NaN,NaN,NaN,X,NaN,NaN


4. Consolidando los archivos de matrices según su información y proceso a ejecutar:

In [60]:
matriz_roles = pd.read_excel(archivo_matriz, sheet_name="Opciones VS Roles", header=3)
matriz_roles.columns

Index(['COD OPCION', 'OPCION', 'DETALLE DE LA OPCION', 'OPCION SENSIBLE',
       'ADMINISTRADOR', 'APROBADOR GENERAL', 'REVISORIA GLOBAL', 'AUDITORIA',
       'SOPORTE_TI', 'PREP_FLUJOS_ADM', 'LANZADOR COMPRAS'],
      dtype='str')

In [59]:
matriz_cargos = pd.read_excel(archivo_matriz, sheet_name="Roles VS Cargos", header=3)
matriz_cargos = matriz_cargos.loc[:, ~matriz_cargos.columns.astype(str).str.contains("^Unnamed")]
matriz_cargos.columns

Index(['NRO. MATRIZ', 'PERFIL', 'COD PERFIL', 'TIPO ENTIDAD', 'COD CARGO',
       'CARGO', 'COD DIVISION', 'DIVISION', 'COD AREA', 'AREA', 'ESTADO',
       'OBSERVACION'],
      dtype='str')

In [61]:
matriz_cargos.to_csv(landing_dir+'matriz_con_cargos.csv',index=False)
matriz_roles.to_csv(landing_dir+'matriz_con_roles.csv',index=False)

5. Consolidando los archivos de recursos humanos según la información requerida:

In [68]:
idm = pd.read_excel(archivo_empleados_idm)
rrhh = pd.read_excel(archivo_empleados_rrhh)

In [8]:
archivos = glob.glob(raw_dir + '*/*.xlsx')

In [9]:
archivos

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\matrices\\Matriz Roles y Perfiles AdminBusiness V8.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\rrhh\\1 - HAM_24_Acuerdo_Empleados_Activos.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\rrhh\\Colaboradores Migrados a IDM.xlsx']

In [10]:
datos = []
for archivo in sorted(archivos):
    datos.append(pd.read_excel(archivo))

df = pd.concat(datos, ignore_index=True)

In [11]:
df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,CIUDAD,USUARIO_DE_RED,ESTADO_EMPLEADO,MOTIVO_AUSENTISMO,FECHA_FIN,TIPO_CONTRATO,COD_OFICINA,FECHA_CONTRATACION,FECHA_INICIO_AUSENTISMO,FECHA_FIN_AUSENTISMO
0,NaN,Matriz de Roles VS Cargos - AdminBusiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
2,NaN,NRO. MATRIZ,PERFIL,COD PERFIL,TIPO ENTIDAD,COD CARGO,CARGO,COD DIVISION,DIVISION,COD AREA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3,NaN,V08,ADMINISTRADOR,101,Interno,1000066,INSPECTOR DE CIERRE,1300000,OPERACIONES,1300005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
4,NaN,V08,ADMINISTRADOR,101,Interno,1000061,SUPERVISOR DE PRODUCCIÓN,1300000,OPERACIONES,1300005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SAN JOSE DE TICLLAS,MZEÑA2,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2021-09-24,NaN,NaN
1304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CHICLAYO,JZULOETA3,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2024-09-04,NaN,NaN
1305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CHICLAYO,WZULOETA9,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2025-01-09,NaN,NaN
1306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ILLIMO,JZUÑIGA7,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2021-09-28,NaN,NaN


**Último paso**: Guardar los datos (union, joins) en un o más archivos, según sea necesario:

In [ ]:
df.to_csv(landing_dir + 'datos_consolidados.csv', index=False)